# Tensor API BatchMatmul 实践

## 概述

本课程为 Tensor API 章节实践课。通过 04.03 与 07.03 的学习，我们已经掌握了 Tensor API 的基本数据通路（GM → L1 → L0A/L0B → mmad → L0C → GM）、Layout/slice/Atom 概念以及高性能矩阵算子的优化手段。本节将通过实现 **BatchMatmul（带 Bias 的批量矩阵乘）** 算子巩固所学知识，重点练习 **batch 维的多核切分**与**分批搬运**。

批量矩阵乘法是普通矩阵乘法在批次维（batch）上的扩展，广泛用于 Beam Search、多路注意力等场景。带 Bias 场景下的核心计算逻辑为：

```text
C[b] = A[b] × B[b] + Bias[b],  b = 0, 1, ..., B - 1
```

不同 batch 的矩阵之间不会互相计算，天然适合按 batch 维做多核切分。**batch 维作为 Layout 的第一维**（无需手工基址偏移）、**L1/L0 两级分批缓冲**（L1_BATCH_SIZE / L0_BATCH_SIZE）、**Bias 经 BiasTable 随路累加**（5 参数 mmad）。

### 前置要求

- 已完成 **04.03 Tensor API 矩阵算子优化实践**，掌握 Tensor API 基本数据通路与事件同步机制（`asc_sync_notify`/`asc_sync_wait`）。
- 已完成 **07.03 Tensor API MxFP4 高性能矩阵算子开发**（选学，本节性能要求为基础可用版）。
- 已配置 CANN 开发环境，并能访问 Ascend 950PR/DT 设备。

### 学习目标

完成本小节后，开发者应能够：

1. 理解 BatchMatmul 的 batch 维多核切分策略（含余数分配）；
2. 掌握由 `block_index` 计算当前核 batch 区间 `[batch_index_start, batch_index_end)` 的方法；
3. 掌握 batch 维作为第一维的 3D Tensor 构造（`nd_ext_layout_ptn`/`nz_layout_ptn`/`zn_layout_ptn`）与 3D slice 批量搬运；
4. 掌握 L1/L0 两级分批缓冲（L1_BATCH_SIZE / L0_BATCH_SIZE）控制片上内存占用；
5. 掌握带 Bias 的 5 参数 `mmad` 调用与 `copy_l1_to_biastable` 搬运通路；
6. 能独立实现 half 输入/输出、FP32 累加、带 Bias 的基础可用 BatchMatmul kernel。

### 运行环境与硬件说明

- 本节样例 **仅支持 Ascend 950PR/DT**，请在 CANNLab 950 尝鲜体验环境中运行。
- learning-hub 的 notebook 在线体验环境和 CANNLab 910B/910C 云开发环境 **不支持** 本节样例。
- **CANN 版本要求：不低于 9.2.0**。

### 本节内容

| 章节 | 内容 | 学习期望 |
| --- | --- | --- |
| 1. 切分策略与数据通路 | batch 维多核切分、L1/L0 分批缓冲、各级 Layout | 理解 BatchMatmul 的切分与分批方案 |
| 2. 样例工程结构 | CMake、Host 侧代码、Python 数据脚本 | 掌握 BatchMatmul 工程的完整结构 |
| 3. 实践 kernel | 补全 7 个 TODO，实现完整 BatchMatmul kernel | 能独立实现 batch 切分与分批搬运 |
| 4. 编译运行验证 | 编译、运行、精度校验 | 验证实现正确性 |
| 5. 参考答案 | 完整参考实现与 TODO 解析 | 对照修正，加深理解 |


---

## 1. 切分策略与数据通路

### 1.1 batch 维多核切分

BatchMatmul 的总任务数 = B 个独立矩阵乘。将 B 个 batch 按顺序均分给所有核心，每个核负责一段**连续**的 batch 区间。

与 04.03 多核版的差异：04.03 在 M/N 方向切分单个大矩阵，而本节直接沿 batch 维切分——每核内的每个 batch 都是一次完整的小矩阵乘（M×K×N = 32×32×32），无需再切分 M/N。切分索引的计算（含余数分配）：

```cpp
uint32_t single_core_b = B / block_num;               // 每核基础 batch 数
uint32_t single_core_res_b = B % block_num;           // 未整除的余数
uint32_t actual_single_core_b = single_core_b + (single_core_res_b > block_index ? 1 : 0);
uint32_t batch_index_start = single_core_b * block_index + min(block_index, single_core_res_b);
uint32_t batch_index_end = actual_single_core_b + batch_index_start;
```

编号相邻的核心访问相邻的 GM 地址，对 L2Cache 友好。

### 1.2 分批数据通路

![数据通路](images/07.09_tensor_api_batch_matmul/data_path.svg)

片上 L1/L0 缓冲无法一次容纳全部 B 个 batch，因此采用**两级分批**：

- **L1 batch 外层循环**：每次用 `copy_gm_to_l1` 从 GM 搬入 `L1_BATCH_SIZE = 32` 个 batch 的 A/B/Bias；
- **L0 batch 内层循环**：每次用 `copy_l1_to_l0a`/`copy_l1_to_l0b`/`copy_l1_to_biastable` 从 L1 加载 `L0_BATCH_SIZE = 4` 个 batch 到 L0A/L0B/BiasTable；
- **mmad 循环**：硬件 mmad 指令不支持 batch 维运算，逐 batch 执行 5 参数 `mmad`（A × B + Bias）；
- **批量搬出**：每完成 `L0_BATCH_SIZE` 个 batch 的计算，用 `copy_l0c_to_gm` 一次将整批结果写回 GM（Fixpipe 同时完成 FP32 → half 转换）。

各级存储的 Layout 约定：

| 数据 | GM | L1 | L0/BiasTable | 说明 |
| --- | --- | --- | --- | --- |
| A `[B, M, K]` | `nd_ext_layout_ptn` | `nz_layout_ptn`（3D） | L0A: `nz_layout_ptn`（3D） | batch 维为第一维 |
| B `[B, K, N]` | `nd_ext_layout_ptn` | `nz_layout_ptn`（3D） | L0B: `zn_layout_ptn`（3D） | Cube 分形要求 |
| Bias `[B, 1, N]` | `nd_ext_layout_ptn` | `nd_ext_layout_ptn`（3D） | BiasTable: `nd_ext_layout_ptn`（3D, FP32） | `__biasbuf__` 存储 |
| C `[B, M, N]` | `nd_ext_layout_ptn` | — | L0C: `nz_layout_ptn`（3D, FP32） | FP32 累加 |

### 1.3 数据规格

本节样例与官方样例保持一致的规格（单 batch 为一个完整 tile）：

| 输入输出 | 数据类型 | Shape | Format |
| --- | --- | --- | --- |
| 输入矩阵 A | half | `[128, 32, 32]` | ND（不转置） |
| 输入矩阵 B | half | `[128, 32, 32]` | ND（不转置） |
| Bias | half | `[128, 1, 32]` | ND |
| 输出矩阵 C | half | `[128, 32, 32]` | ND |

注意：与 04.03 不同，本节 **A/B 均不做转置存储**——batch 维与行列布局统一交由 3D Layout 管理，`copy_gm_to_l1` 搬运时自动完成 ND → NZ/ZN 的格式转换。


---

## 2. 样例工程结构

依次写入工程文件：CMakeLists 与 run.sh、Host 侧公共代码、Python 数据脚本。

### 2.1 CMakeLists.txt 和 run.sh

In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "Tensor API only supports dav-3510 in this sample")

if(NOT CMAKE_ASC_ARCHITECTURES STREQUAL "dav-3510")
    message(FATAL_ERROR "batch_matmul only supports CMAKE_ASC_ARCHITECTURES=dav-3510")
endif()

find_package(ASC REQUIRED)

project(batch_matmul LANGUAGES ASC CXX)

set(ASCEND_HOME $ENV{ASCEND_HOME_PATH})

set(ASC_INCLUDE_DIR "${ASCEND_HOME}/asc/")
include_directories(
    "${ASC_INCLUDE_DIR}"
)

foreach(target_name IN ITEMS
    batch_matmul_practice
    batch_matmul_answer
)
    add_executable(${target_name} ${target_name}.asc)
    target_compile_options(${target_name} PRIVATE
        $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
    )
endforeach()


In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/run.sh
#!/usr/bin/env bash
# CANN 版本要求：不低于 9.2.0（Tensor API asc::te:: 接口自 9.2.0 起提供）
set -euo pipefail

npu_arch="dav-3510"
case_name="practice"

for arg in "$@"; do
    case "$arg" in
        --npu-arch=*)
            npu_arch="${arg#*=}"
            ;;
        --case=*)
            case_name="${arg#*=}"
            ;;
        -h|--help)
            echo "Usage: bash run.sh [--npu-arch=dav-3510] [--case=practice|answer]"
            exit 0
            ;;
    esac
done

if [ "$npu_arch" != "dav-3510" ]; then
    echo "Tensor API BatchMatmul sample only supports --npu-arch=dav-3510"
    exit 1
fi

if [ -z "${ASCEND_HOME_PATH:-}" ]; then
    echo "Please set ASCEND_HOME_PATH before building."
    exit 1
fi

if [ -f "${ASCEND_HOME_PATH}/set_env.sh" ]; then
    # shellcheck disable=SC1091
    source "${ASCEND_HOME_PATH}/set_env.sh"
fi

rm -rf build_out input output
cmake -S . -B build_out -DCMAKE_ASC_ARCHITECTURES="$npu_arch"
python3 scripts/gen_data.py
mkdir -p output

run_one() {
    local name="$1"
    local binary="$2"
    local out_file="output/${name}.bin"
    echo "[BUILD] ${binary}"
    cmake --build build_out -j"$(nproc)" --target "${binary}"
    echo "[RUN] ${name}"
    ./build_out/${binary}
    echo "[VERIFY] ${name}"
    python3 scripts/verify_result.py "${out_file}"
}

case "$case_name" in
    practice)
        run_one practice batch_matmul_practice
        ;;
    answer)
        run_one answer batch_matmul_answer
        ;;
    all)
        run_one practice batch_matmul_practice
        run_one answer batch_matmul_answer
        ;;
    *)
        echo "Unsupported case: ${case_name}"
        echo "Use practice, answer, or all."
        exit 1
        ;;
esac


### 2.2 Host 侧公共代码和数据读写工具

In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/data_utils.h
#ifndef BATCH_MATMUL_DATA_UTILS_H
#define BATCH_MATMUL_DATA_UTILS_H

#include <fcntl.h>
#include <sys/stat.h>
#include <unistd.h>

#include <cstdio>
#include <fstream>
#include <string>

#define ERROR_LOG(fmt, args...) fprintf(stdout, "[ERROR] " fmt "\n", ##args)

inline bool ReadFile(const std::string &filePath, size_t &fileSize, void *buffer, size_t bufferSize)
{
    struct stat sBuf;
    int fileStatus = stat(filePath.data(), &sBuf);
    if (fileStatus == -1) {
        ERROR_LOG("failed to get file: %s", filePath.c_str());
        return false;
    }
    if (S_ISREG(sBuf.st_mode) == 0) {
        ERROR_LOG("%s is not a file", filePath.c_str());
        return false;
    }

    std::ifstream file(filePath, std::ios::binary);
    if (!file.is_open()) {
        ERROR_LOG("open file failed: %s", filePath.c_str());
        return false;
    }

    std::filebuf *buf = file.rdbuf();
    size_t size = buf->pubseekoff(0, std::ios::end, std::ios::in);
    if (size == 0) {
        ERROR_LOG("file size is 0: %s", filePath.c_str());
        return false;
    }
    if (size > bufferSize) {
        ERROR_LOG("file size is larger than buffer size: %s", filePath.c_str());
        return false;
    }

    buf->pubseekpos(0, std::ios::in);
    buf->sgetn(static_cast<char *>(buffer), size);
    fileSize = size;
    return true;
}

inline bool WriteFile(const std::string &filePath, const void *buffer, size_t size)
{
    if (buffer == nullptr) {
        ERROR_LOG("write file failed, buffer is nullptr");
        return false;
    }

    int fd = open(filePath.c_str(), O_RDWR | O_CREAT | O_TRUNC, S_IRUSR | S_IWRITE);
    if (fd < 0) {
        ERROR_LOG("open file failed: %s", filePath.c_str());
        return false;
    }

    size_t writeSize = write(fd, buffer, size);
    (void)close(fd);
    if (writeSize != size) {
        ERROR_LOG("write file failed: %s", filePath.c_str());
        return false;
    }
    return true;
}

#endif


In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/batch_matmul_host.h
#ifndef BATCH_MATMUL_HOST_H
#define BATCH_MATMUL_HOST_H

#include "acl/acl.h"
#include "data_utils.h"

#include <cstdint>
#include <cstdio>
#include <string>

#define CHECK_ACL_RET(expr)                                      \
    do {                                                         \
        aclError ret = (expr);                                   \
        if (ret != ACL_SUCCESS) {                                \
            ERROR_LOG("%s failed, ret = %d", #expr, ret);        \
            return 1;                                            \
        }                                                        \
    } while (0)

template <typename T, typename LaunchFunc>
int run_batch_matmul_host(
    size_t a_file_size, size_t b_file_size, size_t c_file_size, size_t bias_file_size,
    const char *output_path, LaunchFunc launch)
{
    CHECK_ACL_RET(aclInit(nullptr));
    int32_t device_id = 0;
    CHECK_ACL_RET(aclrtSetDevice(device_id));

    aclrtStream stream = nullptr;
    CHECK_ACL_RET(aclrtCreateStream(&stream));

    uint8_t *a_host = nullptr;
    uint8_t *b_host = nullptr;
    uint8_t *c_host = nullptr;
    uint8_t *bias_host = nullptr;
    T *a_device = nullptr;
    T *b_device = nullptr;
    T *c_device = nullptr;
    T *bias_device = nullptr;

    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&a_host), a_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&b_host), b_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&c_host), c_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&bias_host), bias_file_size));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&a_device), a_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&b_device), b_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&c_device), c_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&bias_device), bias_file_size, ACL_MEM_MALLOC_HUGE_FIRST));

    size_t file_size = a_file_size;
    if (!ReadFile("./input/x1_gm.bin", file_size, a_host, a_file_size)) { return 1; }
    file_size = b_file_size;
    if (!ReadFile("./input/x2_gm.bin", file_size, b_host, b_file_size)) { return 1; }
    file_size = bias_file_size;
    if (!ReadFile("./input/bias.bin", file_size, bias_host, bias_file_size)) { return 1; }

    CHECK_ACL_RET(aclrtMemcpy(a_device, a_file_size, a_host, a_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(b_device, b_file_size, b_host, b_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(bias_device, bias_file_size, bias_host, bias_file_size, ACL_MEMCPY_HOST_TO_DEVICE));

    launch(a_device, b_device, c_device, bias_device, stream);
    CHECK_ACL_RET(aclrtSynchronizeStream(stream));

    CHECK_ACL_RET(aclrtMemcpy(c_host, c_file_size, c_device, c_file_size, ACL_MEMCPY_DEVICE_TO_HOST));
    if (!WriteFile(output_path, c_host, c_file_size)) { return 1; }

    (void)aclrtFree(a_device);
    (void)aclrtFree(b_device);
    (void)aclrtFree(c_device);
    (void)aclrtFree(bias_device);
    (void)aclrtFreeHost(a_host);
    (void)aclrtFreeHost(b_host);
    (void)aclrtFreeHost(c_host);
    (void)aclrtFreeHost(bias_host);
    (void)aclrtDestroyStream(stream);
    (void)aclrtResetDevice(device_id);
    (void)aclFinalize();
    return 0;
}

#endif


### 2.3 Python 输入生成脚本和精度校验脚本

In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/scripts/gen_data.py
import os

import numpy as np

B = 128
M = 32
K = 32
N = 32


def main():
    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    np.random.seed(42)
    # BatchMatmul：C[b] = A[b] * B[b] + Bias[b]，三路输入均为 half、ND 布局（不转置）
    a = np.random.uniform(1, 10, (B, M, K)).astype(np.float16)
    b = np.random.uniform(1, 10, (B, K, N)).astype(np.float16)
    bias = np.random.uniform(1, 10, (B, 1, N)).astype(np.float16)

    a.tofile("input/x1_gm.bin")
    b.tofile("input/x2_gm.bin")
    bias.tofile("input/bias.bin")

    # golden：FP32 计算矩阵乘加 Bias 后转 half（与硬件 L0C FP32 累加 → Fixpipe 转 half 一致）
    golden = np.matmul(a.astype(np.float32), b.astype(np.float32)) + bias.astype(np.float32)
    golden.astype(np.float16).tofile("output/golden.bin")

    print(f"generated BatchMatmul inputs: A[{B},{M},{K}], B[{B},{K},{N}], Bias[{B},1,{N}] (half, ND layout)")
    print(f"golden output saved to output/golden.bin")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/scripts/verify_result.py
import sys

import numpy as np

RELATIVE_TOL = 1e-3
ABSOLUTE_TOL = 1e-3
ERROR_TOL = 1e-3


def main():
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python3 verify_result.py output/<case>.bin")

    output_path = sys.argv[1]
    golden_path = "output/golden.bin"

    output = np.fromfile(output_path, dtype=np.float16).reshape(-1)
    golden = np.fromfile(golden_path, dtype=np.float16).reshape(-1)

    if output.size != golden.size:
        raise SystemExit(f"size mismatch: output {output.size}, golden {golden.size}")

    close_mask = np.isclose(output, golden, rtol=RELATIVE_TOL, atol=ABSOLUTE_TOL, equal_nan=True)
    error_indexes = np.where(close_mask == False)[0]

    for idx in error_indexes[:100]:
        golden_val = float(golden[idx])
        output_val = float(output[idx])
        rdiff = abs(output_val - golden_val) / abs(golden_val) if golden_val != 0 else abs(output_val - golden_val)
        print(f"data index: {idx:06d}, expected: {golden_val:.9f}, actual: {output_val:.9f}, rdiff: {rdiff:.6f}")

    error_ratio = float(error_indexes.size) / golden.size
    print(f"error ratio: {error_ratio:.4f}, tolerance: {ERROR_TOL:.4f}")

    if error_ratio > ERROR_TOL:
        raise SystemExit("verify failed!")
    print("test pass!")


if __name__ == "__main__":
    main()


---

## 3. 实践 kernel

`batch_matmul_practice_kernel.h` 中共有 7 个 TODO，其中 **TODO(3)（L1/L0 缓冲 Tensor）为脚手架（已给全，无需修改）**，其余 6 处需补全：

| TODO | 位置 | 考察点 |
| --- | --- | --- |
| (1) | batch 切分索引 | `block_index` → `[batch_index_start, batch_index_end)` 映射（含余数分配） |
| (2) | GM Tensor 构造 | 3D `nd_ext_layout_ptn`（batch 维为第一维） |
| (3) | L1/L0 缓冲 Tensor | 脚手架，无需修改（理解各级 3D 布局即可） |
| (4) | GM→L1 批量搬运 | 3D slice 切出整批 A/B/Bias |
| (5) | L1→L0/BiasTable | 从 L1 批量张量切出 L0 批次 |
| (6) | mmad 循环 | 逐 batch 的 5 参数 mmad（带 Bias、`init_with_zero=true`） |
| (7) | L0C→GM 批量搬出 | GM 目标 batch 起点 = `l1_batch_index + l0_batch_index` |

其中 TODO(2)/(4)/(5) 均已给出 A 的写法作为示例。补全后执行下一节命令验证。

In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/batch_matmul_practice_kernel.h
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#ifndef BATCH_MATMUL_PRACTICE_KERNEL_H
#define BATCH_MATMUL_PRACTICE_KERNEL_H

#include "c_api/asc_simd.h"
#include "tensor_api/tensor.h"

// 注意：本模板包含未补全的 TODO 占位，需补全全部 TODO 后方可编译运行。

template <uint32_t Value, uint32_t Align>
struct ceil_align {
    static constexpr uint32_t value = (Value + Align - 1) / Align * Align;
};

template <
    typename T, uint32_t B, uint32_t M, uint32_t K, uint32_t N,
    uint32_t L1_BATCH_SIZE, uint32_t L0_BATCH_SIZE>
__cube__ __global__ void batch_matmul_kernel(__gm__ T *a, __gm__ T *b, __gm__ T *c, __gm__ T *bias)
{
    asc_init();
    using namespace asc::te;

    // 各级缓冲的单 batch 分形尺寸：L1 A/B 的 C0 为 32/sizeof(T)，L0C/BiasTable 为 float 类型
    constexpr uint32_t C0 = 32 / sizeof(T);
    constexpr uint32_t L1_A_SIZE = ceil_align<M, 16>::value * ceil_align<K, C0>::value;
    constexpr uint32_t L1_B_SIZE = ceil_align<K, 16>::value * ceil_align<N, C0>::value;
    constexpr uint32_t L0_A_SIZE = L1_A_SIZE;
    constexpr uint32_t L0_B_SIZE = ceil_align<K, C0>::value * ceil_align<N, 16>::value;
    constexpr uint32_t L0_C_SIZE = ceil_align<M, 16>::value * ceil_align<N, 16>::value;

    uint32_t block_index = block_idx;

    // TODO(1): 计算 batch 维多核切分索引
    // 提示：B 个 batch 均分给 block_num 个核，余数 B % 核数 逐个分给编号靠前的核
    //   single_core_b = B / block_num                                   // 每核基础 batch 数
    //   single_core_res_b = B % block_num                           // 未整除的余数
    //   actual_single_core_b = single_core_b + (余数分给当前核 ? 1 : 0)  // 当前核实际 batch 数
    //   batch_index_start = single_core_b * block_index + min(block_index, single_core_res_b)  // 起始 batch
    //   batch_index_end = actual_single_core_b + batch_index_start       // 结束 batch（不含）
    uint32_t batch_index_start = 0;  // TODO: 替换为你的计算
    uint32_t batch_index_end = 0;    // TODO: 替换为你的计算

    // TODO(2): 构造 4 个 GM Tensor（batch 维作为第一维的 3D ND 布局，无需手工基址偏移）
    // 提示：A 为 (B, M, K)，B 为 (B, K, N)，C 为 (B, M, N)，Bias 为 (B, 1, N)，均使用 nd_ext_layout_ptn
    //   A 已给出作为示例，将 B/C/Bias 从 2D 布局升级为带 batch 维的 3D 布局
    auto gm_a = make_tensor(make_mem_ptr(a), make_frame_layout<nd_ext_layout_ptn>(B, M, K));
    auto gm_b = make_tensor(make_mem_ptr(b), make_frame_layout<nd_ext_layout_ptn>(K, N));    // TODO(2): 改为 3D
    auto gm_c = make_tensor(make_mem_ptr(c), make_frame_layout<nd_ext_layout_ptn>(M, N));    // TODO(2): 改为 3D
    auto gm_bias = make_tensor(make_mem_ptr(bias), make_frame_layout<nd_ext_layout_ptn>(1, N)); // TODO(2): 改为 3D

    // TODO(3): L1/L0 缓冲 Tensor（脚手架，已给全，无需修改）
    // 说明：L1A/L0A 用 nz_layout_ptn、L1B/L0B 用 zn_layout_ptn（Cube 核分形要求）；
    //   L0C 为 float 类型（FP32 累加）；Bias 在 L1 与 BiasTable 均为 nd_ext_layout_ptn；
    //   各张量第一维均为 batch 维（L1_BATCH_SIZE / L0_BATCH_SIZE）
    __cbuf__ T l1_a_buf[L1_BATCH_SIZE * L1_A_SIZE];
    __cbuf__ T l1_b_buf[L1_BATCH_SIZE * L1_B_SIZE];
    __cbuf__ T l1_bias_buf[L1_BATCH_SIZE * N];
    __ca__ T l0_a_buf[L0_BATCH_SIZE * L0_A_SIZE];
    __cb__ T l0_b_buf[L0_BATCH_SIZE * L0_B_SIZE];
    __cc__ float l0_c_buf[L0_BATCH_SIZE * L0_C_SIZE];
    __biasbuf__ float l0_bias_buf[L0_BATCH_SIZE * N];

    auto l1_a_tensor = make_tensor(make_mem_ptr(l1_a_buf), make_frame_layout<nz_layout_ptn, T>(L1_BATCH_SIZE, M, K));
    auto l1_b_tensor = make_tensor(make_mem_ptr(l1_b_buf), make_frame_layout<nz_layout_ptn, T>(L1_BATCH_SIZE, K, N));
    auto l1_bias_tensor = make_tensor(make_mem_ptr(l1_bias_buf), make_frame_layout<nd_ext_layout_ptn, T>(L1_BATCH_SIZE, 1, N));
    auto l0_a_tensor = make_tensor(make_mem_ptr(l0_a_buf), make_frame_layout<nz_layout_ptn, T>(L0_BATCH_SIZE, M, K));
    auto l0_b_tensor = make_tensor(make_mem_ptr(l0_b_buf), make_frame_layout<zn_layout_ptn, T>(L0_BATCH_SIZE, K, N));
    auto l0_c_tensor = make_tensor(make_mem_ptr(l0_c_buf), make_frame_layout<nz_layout_ptn>(L0_BATCH_SIZE, M, N));
    auto l0_bias_tensor = make_tensor(make_mem_ptr(l0_bias_buf), make_frame_layout<nd_ext_layout_ptn>(L0_BATCH_SIZE, 1, N));

    auto copy_gm_to_l1_atom = make_copy(copy_gm_to_l1{}, gm_to_l1_trait_default{});
    auto copy_l1_to_l0a_atom = make_copy(copy_l1_to_l0a{}, l1_to_l0a_trait_default{});
    auto copy_l1_to_l0b_atom = make_copy(copy_l1_to_l0b{}, l1_to_l0b_trait_default{});
    auto copy_l1_to_bt_atom = make_copy(copy_l1_to_biastable{}, l1_to_biastable_trait_default{});
    auto copy_l0c_to_gm_atom = make_copy(copy_l0c_to_gm{}, l0c_to_gm_trait_default{});
    auto mmad_atom = make_mmad(mmad_operation{}, mmad_trait_default{});

    // 事件同步（脚手架，无需修改）：MTE1/MTE2 管 L1 批量搬运，M/MTE1 管 L0 批量加载，FIX/M 管 L0C 写出
    asc_sync_notify(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
    asc_sync_notify(PIPE_M, PIPE_MTE1, EVENT_ID0);
    asc_sync_notify(PIPE_FIX, PIPE_M, EVENT_ID0);

    // ---- L1 batch 外层循环：每次从 GM 搬入 L1_BATCH_SIZE 个 batch ----
    for (uint32_t l1_batch_index = batch_index_start; l1_batch_index < batch_index_end;
         l1_batch_index += L1_BATCH_SIZE) {
        asc_sync_wait(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
        uint32_t l1_batch_size = min(L1_BATCH_SIZE, batch_index_end - l1_batch_index);

        // TODO(4): 将本批 A/B/Bias 从 GM 搬入 L1（用 3D slice 从 GM 张量切出整批数据）
        // 提示：A 已给出作为示例；coord 第一维为 l1_batch_index，内层为 make_coord(0, 0)；
        //   shape 第一维为 l1_batch_size，内层为单个矩阵的 make_shape(M, K) / (K, N) / (1, N)
        copy(
            copy_gm_to_l1_atom, l1_a_tensor,
            gm_a.slice(make_coord(l1_batch_index, make_coord(0, 0)), make_shape(l1_batch_size, make_shape(M, K))));
        copy(copy_gm_to_l1_atom, l1_b_tensor, gm_b);
        copy(copy_gm_to_l1_atom, l1_bias_tensor, gm_bias);

        asc_sync_notify(PIPE_MTE2, PIPE_MTE1, EVENT_ID0);
        asc_sync_wait(PIPE_MTE2, PIPE_MTE1, EVENT_ID0);

        // ---- L0 batch 内层循环：每次从 L1 加载 L0_BATCH_SIZE 个 batch 到 L0A/L0B/BiasTable ----
        for (uint32_t l0_batch_index = 0; l0_batch_index < l1_batch_size; l0_batch_index += L0_BATCH_SIZE) {
            asc_sync_wait(PIPE_M, PIPE_MTE1, EVENT_ID0);
            uint32_t l0_batch_size = min(L0_BATCH_SIZE, l1_batch_size - l0_batch_index);

            // TODO(5): 将本 L0 批次的 A/B/Bias 从 L1 加载到 L0A/L0B/BiasTable
            // 提示：从 L1 批量张量中切出 L0 批次（A 已给出作为示例）；
            //   coord 第一维为 l0_batch_index，shape 第一维为 l0_batch_size；
            //   A/B 走 copy_l1_to_l0a/l0b，Bias 走 copy_l1_to_biastable
            copy(
                copy_l1_to_l0a_atom, l0_a_tensor,
                l1_a_tensor.slice(
                    make_coord(l0_batch_index, make_coord(0, 0)), make_shape(l0_batch_size, make_shape(M, K))));
            copy(copy_l1_to_l0b_atom, l0_b_tensor, l1_b_tensor);
            copy(copy_l1_to_bt_atom, l0_bias_tensor, l1_bias_tensor);

            asc_sync_notify(PIPE_MTE1, PIPE_M, EVENT_ID0);
            asc_sync_wait(PIPE_MTE1, PIPE_M, EVENT_ID0);
            asc_sync_wait(PIPE_FIX, PIPE_M, EVENT_ID0);

            // TODO(6): 逐 batch 执行带 Bias 的 5 参数 mmad
            // 提示：硬件不支持 batch 维矩阵乘，需在 batch 维循环（l0c_batch_index 从 0 到 l0_batch_size）；
            //   每次从 L0 张量切出第 l0c_batch_index 个 batch（coord 第一维为 l0c_batch_index，shape 第一维为 1）；
            //   init_with_zero 设为 true（每个 batch 输出独立计算，无需跨 K 累加）
            mmad(
                mmad_atom.with(mmad_params{
                    static_cast<uint16_t>(M), static_cast<uint16_t>(N), static_cast<uint16_t>(K),
                    unit_flag_mode::disable, true}),
                l0_c_tensor, l0_a_tensor, l0_b_tensor, l0_bias_tensor);

            asc_sync_notify(PIPE_M, PIPE_FIX, EVENT_ID0);
            asc_sync_notify(PIPE_M, PIPE_MTE1, EVENT_ID0);
            asc_sync_wait(PIPE_M, PIPE_FIX, EVENT_ID0);

            // TODO(7): 将整批 L0_BATCH_SIZE 个 L0C 结果搬出到 GM
            // 提示：GM 目标位置的 batch 起点 = l1_batch_index + l0_batch_index，
            //   shape 第一维为 l0_batch_size，内层为 make_shape(M, N)；源为整个 l0_c_tensor
            copy(copy_l0c_to_gm_atom, gm_c, l0_c_tensor);

            asc_sync_notify(PIPE_FIX, PIPE_M, EVENT_ID0);
        }
        asc_sync_notify(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
    }
    asc_sync_wait(PIPE_M, PIPE_MTE1, EVENT_ID0);
    asc_sync_wait(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
    asc_sync_wait(PIPE_FIX, PIPE_M, EVENT_ID0);
    asc_sync_pipe(PIPE_ALL);
}

#endif


In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/batch_matmul_practice.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#include "batch_matmul_host.h"
#include "batch_matmul_practice_kernel.h"

namespace practice_config {
using T = half;
constexpr uint32_t B = 128;
constexpr uint32_t M = 32;
constexpr uint32_t K = 32;
constexpr uint32_t N = 32;
constexpr uint32_t L1_BATCH_SIZE = 32;
constexpr uint32_t L0_BATCH_SIZE = 4;
constexpr uint32_t NUM_BLOCKS = 32;
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace practice_config;

    constexpr size_t a_file_size = static_cast<size_t>(B) * M * K * sizeof(T);
    constexpr size_t b_file_size = static_cast<size_t>(B) * K * N * sizeof(T);
    constexpr size_t c_file_size = static_cast<size_t>(B) * M * N * sizeof(T);
    constexpr size_t bias_file_size = static_cast<size_t>(B) * N * sizeof(T);

    auto launch = [](T *a, T *b, T *c, T *bias, aclrtStream stream) {
        batch_matmul_kernel<T, B, M, K, N, L1_BATCH_SIZE, L0_BATCH_SIZE>
            <<<NUM_BLOCKS, 0, stream>>>(a, b, c, bias);
    };
    return run_batch_matmul_host<T>(a_file_size, b_file_size, c_file_size, bias_file_size,
                                 "./output/practice.bin", launch);
}


---

## 4. 编译运行验证

补全 TODO 后运行下方命令。预期输出依次为 `[GEN]`、`[BUILD]`、`[RUN]`、`[VERIFY]`，最终 `verify_result.py` 输出 `test pass!`。

常见错误现象提示：

- **batch 区间算错**：部分 batch 无人计算（输出残留脏数据）或重复计算（校验失败）；
- **GM→L1 忘写 3D slice**：整批数据未按 batch 起点切片，各核读到别人的 batch；
- **mmad 忘写 batch 循环**：L0C 只有第 0 个 batch 的结果，其余 batch 输出错乱；
- **L0C→GM 的 batch 起点写错**：输出矩阵写串位，校验大规模报错。

In [ ]:
!cd src/07.09_tensor_api_batch_matmul && bash run.sh --case=practice

---

## 5. 参考答案

以下参考实现与官方 `batch_matmul_tensor_api` 样例的推荐写法一致。写入后可直接运行 `--case=answer` 验证。各 TODO 的详细解析见运行命令之后的 `answer.md`。

In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/batch_matmul_answer_kernel.h
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#ifndef BATCH_MATMUL_ANSWER_KERNEL_H
#define BATCH_MATMUL_ANSWER_KERNEL_H

#include "c_api/asc_simd.h"
#include "tensor_api/tensor.h"

template <uint32_t Value, uint32_t Align>
struct ceil_align {
    static constexpr uint32_t value = (Value + Align - 1) / Align * Align;
};

template <
    typename T, uint32_t B, uint32_t M, uint32_t K, uint32_t N,
    uint32_t L1_BATCH_SIZE, uint32_t L0_BATCH_SIZE>
__cube__ __global__ void batch_matmul_kernel(__gm__ T *a, __gm__ T *b, __gm__ T *c, __gm__ T *bias)
{
    asc_init();
    using namespace asc::te;

    // 各级缓冲的单 batch 分形尺寸：L1 A/B 的 C0 为 32/sizeof(T)，L0C/BiasTable 为 float 类型
    constexpr uint32_t C0 = 32 / sizeof(T);
    constexpr uint32_t L1_A_SIZE = ceil_align<M, 16>::value * ceil_align<K, C0>::value;
    constexpr uint32_t L1_B_SIZE = ceil_align<K, 16>::value * ceil_align<N, C0>::value;
    constexpr uint32_t L0_A_SIZE = L1_A_SIZE;
    constexpr uint32_t L0_B_SIZE = ceil_align<K, C0>::value * ceil_align<N, 16>::value;
    constexpr uint32_t L0_C_SIZE = ceil_align<M, 16>::value * ceil_align<N, 16>::value;

    // ---- [Answer-1] batch 维多核切分：每核负责一段连续 batch，余数分给编号靠前的核 ----
    uint32_t block_index = block_idx;
    uint32_t single_core_b = B / block_num;
    uint32_t single_core_res_b = B % block_num;
    uint32_t actual_single_core_b = single_core_b + (single_core_res_b > block_index ? 1 : 0);
    uint32_t batch_index_start = single_core_b * block_index + min(block_index, single_core_res_b);
    uint32_t batch_index_end = actual_single_core_b + batch_index_start;

    // ---- [Answer-2] GM Tensor：batch 维作为第一维的 3D ND 布局，无需手工基址偏移 ----
    auto gm_a = make_tensor(make_mem_ptr(a), make_frame_layout<nd_ext_layout_ptn>(B, M, K));
    auto gm_b = make_tensor(make_mem_ptr(b), make_frame_layout<nd_ext_layout_ptn>(B, K, N));
    auto gm_c = make_tensor(make_mem_ptr(c), make_frame_layout<nd_ext_layout_ptn>(B, M, N));
    auto gm_bias = make_tensor(make_mem_ptr(bias), make_frame_layout<nd_ext_layout_ptn>(B, 1, N));

    // L1/L0/BiasTable 缓冲区（脚手架，无需修改）
    __cbuf__ T l1_a_buf[L1_BATCH_SIZE * L1_A_SIZE];
    __cbuf__ T l1_b_buf[L1_BATCH_SIZE * L1_B_SIZE];
    __cbuf__ T l1_bias_buf[L1_BATCH_SIZE * N];
    __ca__ T l0_a_buf[L0_BATCH_SIZE * L0_A_SIZE];
    __cb__ T l0_b_buf[L0_BATCH_SIZE * L0_B_SIZE];
    __cc__ float l0_c_buf[L0_BATCH_SIZE * L0_C_SIZE];
    __biasbuf__ float l0_bias_buf[L0_BATCH_SIZE * N];

    // ---- [Answer-3] L1/L0 缓冲 Tensor：batch 维为第一维的 3D 布局 ----
    // L1A/L0A 用 nz_layout_ptn，L1B/L0B 用 zn_layout_ptn（Cube 分形要求）；
    // Bias 在 L1 与 BiasTable 均为 nd_ext_layout_ptn，BiasTable 位于 __biasbuf__
    auto l1_a_tensor = make_tensor(make_mem_ptr(l1_a_buf), make_frame_layout<nz_layout_ptn, T>(L1_BATCH_SIZE, M, K));
    auto l1_b_tensor = make_tensor(make_mem_ptr(l1_b_buf), make_frame_layout<nz_layout_ptn, T>(L1_BATCH_SIZE, K, N));
    auto l1_bias_tensor = make_tensor(make_mem_ptr(l1_bias_buf), make_frame_layout<nd_ext_layout_ptn, T>(L1_BATCH_SIZE, 1, N));
    auto l0_a_tensor = make_tensor(make_mem_ptr(l0_a_buf), make_frame_layout<nz_layout_ptn, T>(L0_BATCH_SIZE, M, K));
    auto l0_b_tensor = make_tensor(make_mem_ptr(l0_b_buf), make_frame_layout<zn_layout_ptn, T>(L0_BATCH_SIZE, K, N));
    auto l0_c_tensor = make_tensor(make_mem_ptr(l0_c_buf), make_frame_layout<nz_layout_ptn>(L0_BATCH_SIZE, M, N));
    auto l0_bias_tensor = make_tensor(make_mem_ptr(l0_bias_buf), make_frame_layout<nd_ext_layout_ptn>(L0_BATCH_SIZE, 1, N));

    auto copy_gm_to_l1_atom = make_copy(copy_gm_to_l1{}, gm_to_l1_trait_default{});
    auto copy_l1_to_l0a_atom = make_copy(copy_l1_to_l0a{}, l1_to_l0a_trait_default{});
    auto copy_l1_to_l0b_atom = make_copy(copy_l1_to_l0b{}, l1_to_l0b_trait_default{});
    auto copy_l1_to_bt_atom = make_copy(copy_l1_to_biastable{}, l1_to_biastable_trait_default{});
    auto copy_l0c_to_gm_atom = make_copy(copy_l0c_to_gm{}, l0c_to_gm_trait_default{});
    auto mmad_atom = make_mmad(mmad_operation{}, mmad_trait_default{});

    // 事件同步（脚手架，无需修改）：MTE1/MTE2 管 L1 批量搬运，M/MTE1 管 L0 批量加载，FIX/M 管 L0C 写出
    asc_sync_notify(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
    asc_sync_notify(PIPE_M, PIPE_MTE1, EVENT_ID0);
    asc_sync_notify(PIPE_FIX, PIPE_M, EVENT_ID0);

    // ---- L1 batch 外层循环：每次从 GM 搬入 L1_BATCH_SIZE 个 batch ----
    for (uint32_t l1_batch_index = batch_index_start; l1_batch_index < batch_index_end;
         l1_batch_index += L1_BATCH_SIZE) {
        asc_sync_wait(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
        uint32_t l1_batch_size = min(L1_BATCH_SIZE, batch_index_end - l1_batch_index);

        // ---- [Answer-4] GM -> L1：3D slice 一次搬入整批 A/B/Bias ----
        // coord 的第一维是 batch 起始索引，内层 coord(0, 0) 为矩阵内坐标；
        // shape 第一维为本批 batch 数，内层 shape 为单个矩阵的 (行, 列)
        copy(
            copy_gm_to_l1_atom, l1_a_tensor,
            gm_a.slice(make_coord(l1_batch_index, make_coord(0, 0)), make_shape(l1_batch_size, make_shape(M, K))));
        copy(
            copy_gm_to_l1_atom, l1_b_tensor,
            gm_b.slice(make_coord(l1_batch_index, make_coord(0, 0)), make_shape(l1_batch_size, make_shape(K, N))));
        copy(
            copy_gm_to_l1_atom, l1_bias_tensor,
            gm_bias.slice(
                make_coord(l1_batch_index, make_coord(0, 0)), make_shape(l1_batch_size, make_shape(1, N))));

        asc_sync_notify(PIPE_MTE2, PIPE_MTE1, EVENT_ID0);
        asc_sync_wait(PIPE_MTE2, PIPE_MTE1, EVENT_ID0);

        // ---- L0 batch 内层循环：每次从 L1 加载 L0_BATCH_SIZE 个 batch 到 L0A/L0B/BiasTable ----
        for (uint32_t l0_batch_index = 0; l0_batch_index < l1_batch_size; l0_batch_index += L0_BATCH_SIZE) {
            asc_sync_wait(PIPE_M, PIPE_MTE1, EVENT_ID0);
            uint32_t l0_batch_size = min(L0_BATCH_SIZE, l1_batch_size - l0_batch_index);

            // ---- [Answer-5] L1 -> L0A/L0B/BiasTable：从 L1 批量张量中切出 L0 批次 ----
            copy(
                copy_l1_to_l0a_atom, l0_a_tensor,
                l1_a_tensor.slice(
                    make_coord(l0_batch_index, make_coord(0, 0)), make_shape(l0_batch_size, make_shape(M, K))));
            copy(
                copy_l1_to_l0b_atom, l0_b_tensor,
                l1_b_tensor.slice(
                    make_coord(l0_batch_index, make_coord(0, 0)), make_shape(l0_batch_size, make_shape(K, N))));
            copy(
                copy_l1_to_bt_atom, l0_bias_tensor,
                l1_bias_tensor.slice(
                    make_coord(l0_batch_index, make_coord(0, 0)), make_shape(l0_batch_size, make_shape(1, N))));

            asc_sync_notify(PIPE_MTE1, PIPE_M, EVENT_ID0);
            asc_sync_wait(PIPE_MTE1, PIPE_M, EVENT_ID0);
            asc_sync_wait(PIPE_FIX, PIPE_M, EVENT_ID0);

            // ---- [Answer-6] mmad：硬件不支持 batch 维矩阵乘，逐 batch 执行带 Bias 的 5 参数 mmad ----
            // init_with_zero = true：每个 batch 的输出独立计算，无需跨 K 累加
            for (uint32_t l0c_batch_index = 0; l0c_batch_index < l0_batch_size; l0c_batch_index++) {
                mmad(
                    mmad_atom.with(mmad_params{
                        static_cast<uint16_t>(M), static_cast<uint16_t>(N), static_cast<uint16_t>(K),
                        unit_flag_mode::disable, true}),
                    l0_c_tensor.slice(make_coord(l0c_batch_index, make_coord(0, 0)), make_shape(1, make_shape(M, N))),
                    l0_a_tensor.slice(make_coord(l0c_batch_index, make_coord(0, 0)), make_shape(1, make_shape(M, K))),
                    l0_b_tensor.slice(make_coord(l0c_batch_index, make_coord(0, 0)), make_shape(1, make_shape(K, N))),
                    l0_bias_tensor.slice(
                        make_coord(l0c_batch_index, make_coord(0, 0)), make_shape(1, make_shape(1, N))));
            }

            asc_sync_notify(PIPE_M, PIPE_FIX, EVENT_ID0);
            asc_sync_notify(PIPE_M, PIPE_MTE1, EVENT_ID0);
            asc_sync_wait(PIPE_M, PIPE_FIX, EVENT_ID0);

            // ---- [Answer-7] L0C -> GM：一次搬出整批 L0_BATCH_SIZE 个输出矩阵 ----
            // GM 目标位置 = L1 批起点 + L0 批内偏移
            copy(
                copy_l0c_to_gm_atom,
                gm_c.slice(
                    make_coord(l1_batch_index + l0_batch_index, make_coord(0, 0)),
                    make_shape(l0_batch_size, make_shape(M, N))),
                l0_c_tensor);

            asc_sync_notify(PIPE_FIX, PIPE_M, EVENT_ID0);
        }
        asc_sync_notify(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
    }
    asc_sync_wait(PIPE_M, PIPE_MTE1, EVENT_ID0);
    asc_sync_wait(PIPE_MTE1, PIPE_MTE2, EVENT_ID0);
    asc_sync_wait(PIPE_FIX, PIPE_M, EVENT_ID0);
    asc_sync_pipe(PIPE_ALL);
}

#endif


In [ ]:
%%writefile src/07.09_tensor_api_batch_matmul/batch_matmul_answer.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#include "batch_matmul_host.h"
#include "batch_matmul_answer_kernel.h"

namespace answer_config {
using T = half;
constexpr uint32_t B = 128;
constexpr uint32_t M = 32;
constexpr uint32_t K = 32;
constexpr uint32_t N = 32;
constexpr uint32_t L1_BATCH_SIZE = 32;
constexpr uint32_t L0_BATCH_SIZE = 4;
constexpr uint32_t NUM_BLOCKS = 32;
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace answer_config;

    constexpr size_t a_file_size = static_cast<size_t>(B) * M * K * sizeof(T);
    constexpr size_t b_file_size = static_cast<size_t>(B) * K * N * sizeof(T);
    constexpr size_t c_file_size = static_cast<size_t>(B) * M * N * sizeof(T);
    constexpr size_t bias_file_size = static_cast<size_t>(B) * N * sizeof(T);

    auto launch = [](T *a, T *b, T *c, T *bias, aclrtStream stream) {
        batch_matmul_kernel<T, B, M, K, N, L1_BATCH_SIZE, L0_BATCH_SIZE>
            <<<NUM_BLOCKS, 0, stream>>>(a, b, c, bias);
    };
    return run_batch_matmul_host<T>(a_file_size, b_file_size, c_file_size, bias_file_size,
                                 "./output/answer.bin", launch);
}


In [ ]:
!cd src/07.09_tensor_api_batch_matmul && bash run.sh --case=answer

In [ ]:
!cat answer/07.09_tensor_api_batch_matmul/answer.md